In [ ]:
# End-of-Course Project: RAG-Enhanced Financial Analyst
本项目旨在自动抽取A股上市公司配股预案中的“发行折价率”与“配股比例”，并计算提取准确率（Evals）。

In [ ]:
import os
import json
import pandas as pd
from pydantic import BaseModel, Field
from openai import OpenAI

# 强烈建议：千万不要把真实的 API Key 写死在代码里传到 GitHub！
# 如果在本地运行，建议使用 dotenv；如果在 Colab 运行，使用 userdata
# 这里假设你已经在环境变量中配置好了
API_KEY = os.getenv("API_KEY", "your-api-key-here")
BASE_URL = "https://api.your-provider.com/v1" # 替换为你的大模型 API 地址

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

In [ ]:
## 1. 定义大模型结构化输出契约 (Structured JSON Contract)
为了避免大模型输出冗余文本，这里使用 Pydantic 定义严格的 JSON Schema。

In [ ]:
class PlacementExtraction(BaseModel):
    stock_name: str = Field(description="上市公司的股票简称")
    placement_size: float = Field(description="配股比例，例如10配3，则输出0.3")
    discount_rate: float = Field(description="发行折价率，用小数表示")

def extract_financial_data(text_content: str, stock_name: str):
    prompt = f"""
    你是一个专业的金融分析师。请阅读以下A股上市公司的配股预案节选，
    提取出'配股比例'和'发行折价率'。

    文本内容：{text_content}
    """
    # 调用基础模型，强制 JSON 输出
    response = client.beta.chat.completions.parse(
        model="your-chosen-model", # 替换为你使用的模型名称，如 deepseek-chat 或 gpt-4o-mini
        messages=[
            {"role": "system", "content": "严格按照要求的JSON格式输出提取结果。"},
            {"role": "user", "content": prompt}
        ],
        response_format=PlacementExtraction,
    )
    return response.choices[0].message.parsed

In [ ]:
## 2. 批量处理预案文本 (Batch Extraction)
读取 `data/announcements/` 目录下的文本文件，调用 API 进行信息抽取，并将结果持续追加写入 `evals/model_outputs.jsonl` 中。

In [ ]:
# 读取基准数据集获取需要处理的股票列表
ground_truth = pd.read_csv('../data/上市公司配股实证研究评估集_25家.csv')
output_jsonl_path = '../evals/model_outputs.jsonl'

# 确保目录存在
os.makedirs('../evals', exist_ok=True)

with open(output_jsonl_path, 'w', encoding='utf-8') as f:
    for stock in ground_truth['股票简称']:
        file_path = f'../data/announcements/{stock}.txt'

        # 实际运行时，请确保这25个txt文件已经存在于相应文件夹
        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as txt_file:
                text_content = txt_file.read()

            print(f"正在处理: {stock}...")
            # 真实调用 API
            result = extract_financial_data(text_content, stock)

            # 将结构化结果保存
            f.write(result.model_dump_json() + '\n')
        else:
            print(f"警告：找不到 {stock}.txt，跳过此样本。")

print("批量信息抽取完成！")

In [ ]:
## 3. 自动化评估 (Evals)
将模型生成的 JSONL 结果与人工基准 CSV 进行对比，计算准确率指标。这一步取代了原先模糊的“缩短分析时间”指标。

In [ ]:
def run_evals():
    truth_dict = ground_truth.set_index('股票简称').to_dict('index')

    total_cases = 0
    size_correct = 0
    discount_correct = 0

    with open(output_jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            model_out = json.loads(line)
            stock = model_out['stock_name']

            if stock in truth_dict:
                total_cases += 1
                truth_row = truth_dict[stock]

                # 容差范围设为 0.01 以应对浮点数精度问题
                if abs(model_out['placement_size'] - truth_row['配股比例']) < 0.01:
                    size_correct += 1
                if abs(model_out['discount_rate'] - truth_row['发行折价率']) < 0.01:
                    discount_correct += 1

    print(f"=== Evals 评估报告 ===")
    if total_cases == 0:
        print("暂无测试结果，请确保 API 抽取步骤已成功运行。")
        return

    print(f"总测试样本数: {total_cases}")
    print(f"配股比例 (Placement Size) 提取准确率: {size_correct}/{total_cases} ({size_correct/total_cases*100:.1f}%)")
    print(f"发行折价率 (Discount Rate) 提取准确率: {discount_correct}/{total_cases} ({discount_correct/total_cases*100:.1f}%)")

run_evals()